# Path Parameters in Express

Path parameters are dynamic segments of a URL path marked with a colon (`:`). You capture and read them using the `req.params` object inside your route handler.

## Defining and Accessing Parameters

- **Single parameter** — Define `app.get('/users/:userId', ...)` and read `req.params.userId`.
- **Multiple parameters** — Combine segments like `app.get('/posts/:category/:postId', ...)` and read `req.params.category` and `req.params.postId`.
- **Data type** — All captured path parameters are stored as **strings**, always.

## Quick Code Example

```javascript
const express = require('express');
const app = express();

app.get('/users/:userId/books/:bookId', (req, res) => {
  const { userId, bookId } = req.params;
  res.send(`User: ${userId}, Book: ${bookId}`);
});

app.listen(3000);
```

Destructuring out of `req.params` like this is the conventional style — cleaner than repeating `req.params.x` throughout the handler.

## The String Trap

This is the single most common bug with path parameters:

```javascript
app.get('/users/:id', (req, res) => {
  const { id } = req.params;

  console.log(typeof id);        // "string" — always
  console.log(id === 1);         // false, even for /users/1
  console.log(id + 1);           // "11" for /users/1 — concatenation, not addition

  const numericId = Number(id);  // convert explicitly
});
```

`Number("abc")` returns `NaN` rather than throwing, so conversion alone isn't validation. Guard it:

```javascript
app.get('/users/:id', (req, res) => {
  const id = Number(req.params.id);

  if (!Number.isInteger(id) || id < 1) {
    return res.status(400).json({ error: 'Invalid user ID' });
  }

  // safe to use `id` as a number from here
});
```

Without this, a request to `/users/abc` reaches your database layer with garbage input.

## Constraining What a Parameter Matches

You can attach a regular expression to a parameter so non-matching URLs fall through to the next route instead of entering your handler:

```javascript
// Express 4
app.get('/users/:id(\\d+)', handler);   // only matches digits
```

In **Express 5** this inline-regex syntax was removed. Validate inside the handler (as above), or use a dedicated validation library like `zod` or `express-validator`.

## Optional Parameters

```javascript
// Express 4
app.get('/products/:category?', handler);   // matches /products and /products/books

// Express 5
app.get('/products{/:category}', handler);
```

Either way, check for the value before using it — it will be `undefined` when omitted.

## Route Order Still Applies

A parameter matches *anything* in that position, including words you meant as literal paths:

```javascript
// ❌ /users/new never fires — ":id" captures "new"
app.get('/users/:id', handler);
app.get('/users/new', handler);

// ✅ static segments first
app.get('/users/new', handler);
app.get('/users/:id', handler);
```

## Params vs Query: Which to Use

| | Path parameter | Query parameter |
| --- | --- | --- |
| Looks like | `/users/42` | `/users?page=2` |
| Read via | `req.params.id` | `req.query.page` |
| Purpose | Identifies **which** resource | Modifies **how** you want it |
| Required? | Yes — it's part of the route | Usually optional |
| Good for | Resource IDs, slugs, categories | Filters, sorting, pagination, search terms |

A useful test: if removing it would make the URL meaningless, it's a path parameter. If removing it just gives you the default view, it's a query parameter.

```javascript
// Both together — common in real APIs
// GET /users/42/orders?status=shipped&limit=10
app.get('/users/:userId/orders', (req, res) => {
  const { userId } = req.params;
  const { status, limit = 20 } = req.query;
  // ...
});
```

## `app.param()` for Shared Logic

When several routes need the same lookup, `app.param()` runs once whenever that parameter appears — a clean way to avoid repeating "fetch the user or 404":

```javascript
app.param('userId', async (req, res, next, id) => {
  const user = await db.findUser(Number(id));
  if (!user) return res.status(404).json({ error: 'User not found' });
  req.user = user;   // attach for downstream handlers
  next();
});

// Both routes now get req.user populated automatically
app.get('/users/:userId', (req, res) => res.json(req.user));
app.get('/users/:userId/orders', (req, res) => res.json(req.user.orders));
```

## Nested Routers and `mergeParams`

A router mounted under a parameterised path doesn't inherit the parent's params by default:

```javascript
// app.js
app.use('/users/:userId/posts', postsRouter);

// routes/posts.js
const router = express.Router({ mergeParams: true }); // ← without this, req.params.userId is undefined

router.get('/', (req, res) => {
  res.send(`Posts for user ${req.params.userId}`);
});
```

## References

- [Express — Routing guide](https://expressjs.com/en/guide/routing.html)